# Train XAI YOLO From Scratch

Notebook nay huan luyen YOLO + XAI ngay tu dau tu checkpoint pretrained, khong fine-tune tu baseline da train xong.
Muc tieu la so sanh cong bang giua baseline `model.train(...)` va pipeline `XAI-guided training` voi cung diem khoi tao.


In [1]:
!pip install ultralytics==8.4.61

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.5 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 w

In [2]:
from pathlib import Path

REPO_URL = "https://github.com/Thanhmay2406/xai-driven-saliency-loss.git"
CLONE_DIR = Path("/kaggle/working/xai-driven-saliency-loss")

if not CLONE_DIR.exists():
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print(f"Repo already exists: {CLONE_DIR}")
    %cd {CLONE_DIR}
    !git pull --ff-only


Cloning into '/kaggle/working/xai-driven-saliency-loss'...
remote: Enumerating objects: 428, done.
remote: Counting objects: 100% (428/428), done.
remote: Compressing objects: 100% (295/295), done.
remote: Total 428 (delta 191), reused 362 (delta 125), pack-reused 0 (from 0)
Receiving objects: 100% (428/428), 33.33 MiB | 39.63 MiB/s, done.
Resolving deltas: 100% (191/191), done.


In [3]:
import sys
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working")
DEFAULT_REPO_ROOT = KAGGLE_WORKING / "xai-driven-saliency-loss"
REPO_ROOT = DEFAULT_REPO_ROOT if (DEFAULT_REPO_ROOT / "src").exists() else Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

CFG = {
    "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
    "INIT_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "/kaggle/working/yolo_xai_from_scratch",
    "XAI_RUN_NAME": "xai_from_scratch_v2_fixed",
    "IMGSZ": 640,
    "BATCH": 16,
    "DEVICE": 0,
    "WORKERS": 4,
    "EPOCHS": 60,
    "PATIENCE": 10,
    "SEED": 42,
    "OPTIMIZER": "SGD",
    "LR": 1e-2,
    "LRF": 1e-2,
    "MOMENTUM": 0.937,
    "WEIGHT_DECAY": 5e-4,
    "WARMUP_EPOCHS": 3,
    "XAI_METHOD": "activation",
    "LAMBDA_SALIENCY": 0.006,
    "NUM_CLASSES": 6,
    "USE_PROGRESSIVE_LAMBDA": True,
    "PROGRESSIVE_WARMUP_EPOCHS": 30,
    "GT_MASK_MODE": "soft",
    "SOFT_MASK_SIGMA": 0.35,
    "SHRUNK_BOX_RATIO": 0.7,
}

DATASET_YAML = Path(CFG["DATASET_YAML"])
INIT_WEIGHTS = CFG["INIT_WEIGHTS"]
TRAIN_ARGS = {
    "data": str(DATASET_YAML),
    "project": CFG["PROJECT_DIR"],
    "name": CFG["XAI_RUN_NAME"],
    "epochs": CFG["EPOCHS"],
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
    "workers": CFG["WORKERS"],
    "patience": CFG["PATIENCE"],
    "seed": CFG["SEED"],
}
VAL_ARGS = {
    "data": str(DATASET_YAML),
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
}
print("Dataset YAML:", DATASET_YAML)
print("Init weights:", INIT_WEIGHTS)
print("Train args:", TRAIN_ARGS)


REPO_ROOT: /kaggle/working/xai-driven-saliency-loss
Dataset YAML: /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml
Init weights: yolo11n.pt
Train args: {'data': '/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml', 'project': '/kaggle/working/yolo_xai_from_scratch', 'name': 'xai_from_scratch_v2_fixed', 'epochs': 60, 'imgsz': 640, 'batch': 16, 'device': 0, 'workers': 4, 'patience': 10, 'seed': 42}


In [4]:
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

Path(CFG["PROJECT_DIR"]).mkdir(parents=True, exist_ok=True)


In [5]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Load Initial Pretrained Checkpoint

Cell nay load checkpoint pretrained ban dau (`yolo11n.pt`) de kiem tra diem xuat phat truoc khi train XAI tu dau.


In [6]:
init_model = YOLO(str(INIT_WEIGHTS))
init_model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_

In [7]:
val_results = init_model.val(
    **VAL_ARGS,
    split="val",
)
val_results


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.3 ms, read: 1.6±0.5 MB/s, size: 10.9 KB)
val: Scanning /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels/valid... 1603 images, 289 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1603/1603 159.0it/s 10.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 101/101 7.1it/s 14.2s
                   all       1603       1959    0.00167    0.00301   3.25e-05   8.04e-06
               bicycle        293        293    0.00665     0.0102   0.000152   3.91e-05
                   car        352        409          0          0          0          0
            motorcycle        469   

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7aebe42189e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [8]:
test_results = init_model.val(
    **VAL_ARGS,
    split="test",
)
test_results

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.3±0.3 ms, read: 1.5±0.4 MB/s, size: 9.7 KB)
val: Scanning /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels/test... 1562 images, 335 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1562/1562 172.3it/s 9.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 98/98 7.6it/s 12.8s
                   all       1562       1780    0.00036   0.000932   2.22e-06   2.22e-07
               bicycle        258        258          0          0          0          0
                   car        280        302          0          0          0          0
            motorcycle        434        449          0          0          0          0
              airplane        306  

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7aeb6e3ae1e0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

## XAI Training From Scratch

Pha nay huan luyen YOLO + XAI ngay tu checkpoint pretrained `yolo11n.pt`, khong di qua buoc fine-tune tu baseline da co.
De so sanh cong bang voi baseline, notebook nay giu cung `TRAIN_ARGS` va custom loop chi khac o viec cong them `saliency_loss`.
Vi XAI dang dung custom loop voi `train_loader`, early stopping phai duoc theo doi thu cong sau moi epoch.

Training trong repo hien tai phai dung `activation` de `saliency_loss` thuc su co gradient.
`Grad-CAM`/`Grad-CAM++`/`EigenCAM` chi dung cho visualization hoac evaluation offline.


In [9]:
import inspect
from src.training import (
    UltralyticsYOLOXAIDetectionTrainer,
    UltralyticsYOLOXAITrainerConfig,
    train_ultralytics_yolo_xai,
)

xai_cfg = UltralyticsYOLOXAITrainerConfig(
    xai_method=CFG["XAI_METHOD"],
    lambda_saliency=CFG["LAMBDA_SALIENCY"],
    num_classes=CFG["NUM_CLASSES"],
    use_progressive_lambda=CFG["USE_PROGRESSIVE_LAMBDA"],
    progressive_warmup_epochs=CFG["PROGRESSIVE_WARMUP_EPOCHS"],
    gt_mask_mode=CFG["GT_MASK_MODE"],
    soft_mask_sigma=CFG["SOFT_MASK_SIGMA"],
    shrunk_box_ratio=CFG["SHRUNK_BOX_RATIO"],
)
print("Trainer source:", inspect.getfile(UltralyticsYOLOXAIDetectionTrainer))
xai_cfg

Trainer source: /kaggle/working/xai-driven-saliency-loss/src/training/ultralytics_xai_detection_trainer.py


UltralyticsYOLOXAITrainerConfig(xai_method='activation', lambda_saliency=0.006, num_classes=6, target_layer=None, prefer_branch='small', use_progressive_lambda=True, progressive_warmup_epochs=30, gt_mask_mode='soft', soft_mask_sigma=0.35, shrunk_box_ratio=0.7)

In [10]:
import json
import random

import numpy as np
import pandas as pd
import torch

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

train_overrides = {
    **TRAIN_ARGS,
    "model": str(INIT_WEIGHTS),
    "optimizer": CFG["OPTIMIZER"],
    "lr0": CFG["LR"],
    "lrf": CFG["LRF"],
    "momentum": CFG["MOMENTUM"],
    "weight_decay": CFG["WEIGHT_DECAY"],
    "warmup_epochs": CFG["WARMUP_EPOCHS"],
    "plots": True,
    "val": True,
}

trainer = train_ultralytics_yolo_xai(
    train_overrides=train_overrides,
    xai_config=xai_cfg,
)

run_dir = Path(trainer.save_dir)
summary_path = run_dir / "summary" / "run_summary.json"
history_path = run_dir / "weights" / "train_history.json"
history_csv_path = run_dir / "weights" / "train_history.csv"
print("Run dir:", run_dir)
print("Summary:", summary_path)
print("History:", history_path)
print("History CSV:", history_csv_path)

run_summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(run_summary, indent=2, ensure_ascii=False))
run_summary


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=xai_from_scratch_v2_fixed, nbs=64, nms=False, opset=Non

{'best_epoch': 47,
 'best_epoch_by_fitness': 47,
 'best_epoch_by_map50_95': 47,
 'best_val_map50_95': 0.39771055366772784,
 'num_epochs_completed': 58,
 'best_checkpoint': '/kaggle/working/yolo_xai_from_scratch/xai_from_scratch_v2_fixed/weights/best.pt',
 'last_checkpoint': '/kaggle/working/yolo_xai_from_scratch/xai_from_scratch_v2_fixed/weights/last.pt',
 'history_csv': '/kaggle/working/yolo_xai_from_scratch/xai_from_scratch_v2_fixed/weights/train_history.csv',
 'history_json': '/kaggle/working/yolo_xai_from_scratch/xai_from_scratch_v2_fixed/weights/train_history.json',
 'best_val_metrics': {'precision': 0.7380272225698289,
  'recall': 0.7210019138444344,
  'map50': 0.7579263796138647,
  'map50_95': 0.39771055366772784,
  'fitness': 0.39771055366772784,
  'per_class_map50_95': {'drill': 0.39771055366772784,
   'Broken': 0.6219760535472634,
   'Chipped': 0.3195285698639806,
   'Scratched': 0.3248396561007081,
   'Severe_Rust': 0.31664857441700556,
   'Tip_Wear': 0.40555991440968153},
 

In [11]:
history_df = pd.read_csv(history_csv_path)
display(history_df)

metric_cols = [
    "train_saliency_loss",
    "train_energy_in_box",
    "train_pointing_game",
    "train_saliency_iou",
]

diagnostics = []
for col in metric_cols:
    values = history_df[col].tolist()
    diagnostics.append({
        "metric": col,
        "unique_count": int(history_df[col].nunique(dropna=False)),
        "first": values[0],
        "last": values[-1],
        "values": values,
    })

diag_df = pd.DataFrame(diagnostics)
display(diag_df)

if int(run_summary.get("num_epochs_completed", 0)) != len(history_df):
    raise AssertionError(
        f"run_summary num_epochs_completed={run_summary.get('num_epochs_completed')} khong khop so dong train_history={len(history_df)}"
    )

frozen_metrics = [item["metric"] for item in diagnostics if item["unique_count"] <= 1]
if frozen_metrics:
    raise AssertionError(f"Cac metric XAI dang dung im: {frozen_metrics}")

if float(history_df["train_saliency_loss"].sum()) <= 0.0:
    raise AssertionError("train_saliency_loss van bang 0 tren toan bo run")

print("Artifact check passed.")


,epoch,lr,lambda_saliency,train_detection_loss,train_saliency_loss,train_total_loss,train_energy_in_box,train_pointing_game,train_saliency_iou,val_precision,val_recall,val_map50,val_map50_95,val_fitness,epoch_seconds
0,1,0.003326,0.0000,8.24000,0.984078,8.240000,0.015922,0.068067,0.029105,0.347550,0.24991,0.205430,0.088690,0.08869,139.698645
1,2,0.006549,0.0002,6.64342,0.985950,6.643617,0.014050,0.044960,0.018940,0.247380,0.35106,0.214700,0.084440,0.08444,113.371320
2,3,0.009663,0.0004,6.48282,0.987062,6.483215,0.012938,0.043229,0.017510,0.260840,0.27944,0.200660,0.073690,0.07369,113.273015
3,4,0.009505,0.0006,6.49540,0.987560,6.495993,0.012440,0.048879,0.018969,0.280690,0.37580,0.239810,0.092860,0.09286,113.115079
4,5,0.009340,0.0008,6.20991,0.987756,6.210700,0.012244,0.039642,0.015394,0.377530,0.37428,0.314920,0.136240,0.13624,112.762859
5,6,0.009175,0.0010,5.98957,0.988256,5.990558,0.011744,0.055993,0.016699,0.381520,0.42281,0.341760,0.146470,0.14647,113.187244
6,7,0.009010,0.0012,5.78965,0.988351,5.790836,0.011649,0.043478,0.016503,0.418570,0.40185,0.374440,0.172190,0.17219,113.044910
7,8,0.008845,0.0014,5.68629,0.989238,5.687675,0.010762,0.038251,0.013057,0.517950,0.49117,0.486730,0.221640,0.22164,113.167856
8,9,0.008680,0.0016,5.55197,0.988492,5.553552,0.011508,0.039280,0.012908,0.456580,0.51851,0.473560,0.218700,0.21870,113.100856
9,10,0.008515,0.0018,5.41529,0.988983,5.417070,0.011017,0.040095,0.012456,0.546640,0.50294,0.510960,0.241510,0.24151,113.035680


,metric,unique_count,first,last,values
0,train_saliency_loss,57,0.984078,0.987546,"[0.984078113493528, 0.9859499409299204, 0.9870..."
1,train_energy_in_box,57,0.015922,0.012454,"[0.0159218872189026, 0.0140500586137109, 0.012..."
2,train_pointing_game,57,0.068067,0.088047,"[0.0680665197293866, 0.0449596378492346, 0.043..."
3,train_saliency_iou,57,0.029105,0.019077,"[0.029104717243091, 0.0189395793698141, 0.0175..."


Artifact check passed.


In [12]:
!rm -rf /kaggle/working/xai-driven-saliency-loss